# Policy RAG Pipeline — Ingestion to Retrieval (LangChain + Azure OpenAI + ChromaDB)

Single-notebook version of the pipeline: load PDFs → chunk → embed → Chroma vector store → semantic retrieval → grounded answer generation.

**Security note:** this notebook reads credentials from a `.env` file sitting next to it (via `python-dotenv`) — it does **not** hardcode the API key in any cell. Keep `.env` out of version control and rotate the key if it's ever pasted into a chat, doc, or screenshot.

**Requirements:** `pip install langchain langchain-openai langchain-community langchain-chroma chromadb pypdf python-dotenv tiktoken`


## 1. Setup — imports, config, and environment variables

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env in the current working directory

# --- Paths ---
DATA_DIR = Path("./data")                  # folder containing the source PDFs
CHROMA_DIR = Path(os.getenv("CHROMA_DIR", "./chroma_db"))
CHROMA_COLLECTION_NAME = os.getenv("CHROMA_COLLECTION_NAME", "policy_docs")

# --- Azure OpenAI settings (loaded from .env, never hardcoded here) ---
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

# These are Azure *deployment names* (Azure AI Studio) — may differ from base model names
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1")
AZURE_OPENAI_CHAT_DEPLOYMENT_SECONDARY = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

required = {
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_CHAT_DEPLOYMENT": AZURE_OPENAI_CHAT_DEPLOYMENT,
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT": AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required .env variables: {missing}")

print("Config loaded OK")
print("Chat deployment:", AZURE_OPENAI_CHAT_DEPLOYMENT)
print("Embedding deployment:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT)


## 2. Ingestion — load PDFs and split into semantic chunks

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_documents(data_dir: Path):
    pdf_paths = sorted(data_dir.glob("*.pdf"))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in {data_dir}")
    documents = []
    for path in pdf_paths:
        pages = PyPDFLoader(str(path)).load()
        for page in pages:
            page.metadata["source_file"] = path.name
        documents.extend(pages)
        print(f"Loaded {len(pages)} page(s) from {path.name}")
    return documents

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=["\nSECTION", "\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    return chunks

documents = load_documents(DATA_DIR)
chunks = split_documents(documents)
chunks[0]


## 3. Embedding + Vector Store — build and persist the Chroma collection

In [ ]:
import shutil
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
)

# Wipe any previous collection so re-running this cell doesn't duplicate chunks
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=CHROMA_COLLECTION_NAME,
    persist_directory=str(CHROMA_DIR),
)
print(f"Chroma collection '{CHROMA_COLLECTION_NAME}' persisted to {CHROMA_DIR}")
print("Document count:", vector_store._collection.count())


> Once you've built the collection, you can skip step 2/3 on future runs and just reload it:
> ```python
> vector_store = Chroma(
>     collection_name=CHROMA_COLLECTION_NAME,
>     embedding_function=embeddings,
>     persist_directory=str(CHROMA_DIR),
> )
> ```


## 4. Retrieval — semantic search over the index

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

def format_docs(docs) -> str:
    blocks = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page", "?")
        blocks.append(f"[{source} | page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)

# quick sanity check of retrieval on its own, before adding generation
test_docs = retriever.invoke("How many physical therapy visits before prior authorization on Silver HMO?")
print(format_docs(test_docs))


## 5. Generation — grounded answer synthesis with Azure Chat OpenAI

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

SYSTEM_PROMPT = """You are a benefits policy assistant. Answer ONLY using the
provided context excerpts from the policy documents. Each excerpt is tagged
with its source document and page.

Rules:
- If the answer isn't in the context, say you don't have that information in
  the knowledge base — do not guess.
- When plan-specific rules differ (e.g. Gold PPO vs Silver HMO), state the
  difference explicitly rather than giving one blended answer.
- Always cite which document(s) support your answer, e.g. (Source: GOLD-PPO-2026).
- Be concise and precise; this is used for benefits/authorization decisions.

Context:
{context}
"""

def build_chain(deployment: str = None):
    llm = AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_deployment=deployment or AZURE_OPENAI_CHAT_DEPLOYMENT,
        temperature=0,
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", SYSTEM_PROMPT), ("human", "{question}")]
    )
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

rag_chain = build_chain()


## 6. Ask questions

In [ ]:
def ask(question: str, chain=rag_chain):
    docs = retriever.invoke(question)
    answer = chain.invoke(question)
    print("ANSWER:\n", answer)
    print("\nSOURCES:")
    for d in docs:
        print(f"  - {d.metadata.get('source_file')} (page {d.metadata.get('page')})")
    return answer

ask("How many physical therapy visits are allowed before prior authorization is required, for Gold PPO vs Silver HMO?")


In [ ]:
# Try more questions
ask("Does emergency room imaging need prior authorization?")


In [ ]:
# Optional: run the same question against the secondary deployment (e.g. gpt-5-mini)
secondary_chain = build_chain(deployment=AZURE_OPENAI_CHAT_DEPLOYMENT_SECONDARY)
ask("What is the provider appeal filing window after a claim denial?", chain=secondary_chain)
